In [1]:
# First, uninstall the current version
!pip uninstall matplotlib -y

# Install matplotlib 3.8.4 (last stable version before 3.9)
!pip install matplotlib==3.8.4

# Verify the installation
import matplotlib
print(f"Matplotlib version: {matplotlib.__version__}")

# Restart the kernel after installation for changes to take effect
# In Deepnote: Kernel → Restart

Found existing installation: matplotlib 3.8.4
Uninstalling matplotlib-3.8.4:
  Successfully uninstalled matplotlib-3.8.4
  Using cached matplotlib-3.8.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (11.6 MB)

[notice] A new release of pip is available: 23.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Matplotlib version: 3.8.4


In [2]:
#!/usr/bin/env python3
"""
5_MACHINE_LEARNING.py

Machine learning analysis comparing three approaches to predicting pluralistic ignorance:
1. LLM predictions (GPT, Claude, Gemini, Llama, Ensemble)
2. OLS regression (traditional features)
3. Lasso regression (regularized ML approach)

Evaluation: 10 repeated 80:20 train-test splits, stratified by continent
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_rel
import warnings
warnings.filterwarnings('ignore')

# Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 10)

print("="*80)
print("ML COMPARISON: LLMs (Stage 8) vs OLS vs LASSO (Actual Data)")
print("="*80)
print("\nComparing three approaches to predict pluralistic ignorance:")
print("  1. LLM predictions from Stage 8 (GPT, Claude, Gemini, Llama, Ensemble)")
print("  2. OLS regression trained on actual country features & outcomes")
print("  3. Lasso regression trained on actual country features & outcomes")
print("\nEvaluation: 10 repeated 80:20 train-test splits (stratified by continent)")

# ================================================================
# 1. Load and Prepare Data
# ================================================================

print("\n" + "="*80)
print("1. LOADING DATA")
print("="*80)

df = pd.read_csv("predictions_all_stages_long.csv")
model_cols = [col for col in df.columns if col.startswith('pred_')]
df['pred_ensemble'] = df[model_cols].mean(axis=1)

# Add continent mapping
continent_mapping = {
    'Afghanistan': 'Asia', 'Albania': 'Europe', 'Algeria': 'Africa', 'Argentina': 'South America',
    'Armenia': 'Asia', 'Australia': 'Oceania', 'Austria': 'Europe', 'Bangladesh': 'Asia',
    'Belgium': 'Europe', 'Benin': 'Africa', 'Bolivia': 'South America', 'Bosnia Herzegovina': 'Europe',
    'Botswana': 'Africa', 'Brazil': 'South America', 'Bulgaria': 'Europe', 'Burkina Faso': 'Africa',
    'Cambodia': 'Asia', 'Cameroon': 'Africa', 'Canada': 'North America', 'Chad': 'Africa',
    'Chile': 'South America', 'China': 'Asia', 'Colombia': 'South America', 'Congo Brazzaville': 'Africa',
    'Costa Rica': 'North America', 'Croatia': 'Europe', 'Cyprus': 'Europe', 'Czech Republic': 'Europe',
    'Denmark': 'Europe', 'Dominican Republic': 'North America', 'Ecuador': 'South America', 'Egypt': 'Africa',
    'El Salvador': 'North America', 'Estonia': 'Europe', 'Ethiopia': 'Africa', 'Finland': 'Europe',
    'France': 'Europe', 'Gabon': 'Africa', 'Georgia': 'Asia', 'Germany': 'Europe',
    'Ghana': 'Africa', 'Greece': 'Europe', 'Guatemala': 'North America', 'Guinea': 'Africa',
    'Haiti': 'North America', 'Honduras': 'North America', 'Hong Kong': 'Asia', 'Hungary': 'Europe',
    'Iceland': 'Europe', 'India': 'Asia', 'Indonesia': 'Asia', 'Iran': 'Asia',
    'Iraq': 'Asia', 'Ireland': 'Europe', 'Israel': 'Asia', 'Italy': 'Europe',
    'Ivory Coast': 'Africa', 'Jamaica': 'North America', 'Japan': 'Asia', 'Jordan': 'Asia',
    'Kazakhstan': 'Asia', 'Kenya': 'Africa', 'Kosovo': 'Europe', 'Kyrgyzstan': 'Asia',
    'Laos': 'Asia', 'Latvia': 'Europe', 'Lebanon': 'Asia', 'Liberia': 'Africa', 'Libya': 'Africa',
    'Lithuania': 'Europe', 'Luxembourg': 'Europe', 'Macedonia': 'Europe', 'Madagascar': 'Africa',
    'Malawi': 'Africa', 'Malaysia': 'Asia', 'Mali': 'Africa', 'Malta': 'Europe',
    'Mauritania': 'Africa', 'Mauritius': 'Africa', 'Mexico': 'North America', 'Moldova': 'Europe',
    'Mongolia': 'Asia', 'Montenegro': 'Europe', 'Morocco': 'Africa', 'Mozambique': 'Africa',
    'Myanmar': 'Asia', 'Namibia': 'Africa', 'Nepal': 'Asia', 'Netherlands': 'Europe',
    'New Zealand': 'Oceania', 'Nicaragua': 'North America', 'Niger': 'Africa', 'Nigeria': 'Africa',
    'North Macedonia': 'Europe', 'Norway': 'Europe', 'Pakistan': 'Asia', 'Palestinian Territories': 'Asia',
    'Panama': 'North America', 'Paraguay': 'South America', 'Peru': 'South America', 'Philippines': 'Asia',
    'Poland': 'Europe', 'Portugal': 'Europe', 'Romania': 'Europe', 'Russia': 'Europe', 'Rwanda': 'Africa',
    'Saudi Arabia': 'Asia', 'Senegal': 'Africa', 'Serbia': 'Europe', 'Sierra Leone': 'Africa',
    'Singapore': 'Asia', 'Slovakia': 'Europe', 'Slovenia': 'Europe', 'South Africa': 'Africa',
    'South Korea': 'Asia', 'Spain': 'Europe', 'Sri Lanka': 'Asia', 'Sweden': 'Europe',
    'Switzerland': 'Europe', 'Taiwan': 'Asia', 'Tajikistan': 'Asia', 'Tanzania': 'Africa',
    'Thailand': 'Asia', 'Togo': 'Africa', 'Tunisia': 'Africa', 'Turkey': 'Asia',
    'Turkmenistan': 'Asia', 'Uganda': 'Africa', 'Ukraine': 'Europe', 'United Arab Emirates': 'Asia',
    'United Kingdom': 'Europe', 'United States': 'North America', 'Uruguay': 'South America',
    'Uzbekistan': 'Asia', 'Venezuela': 'South America', 'Vietnam': 'Asia', 'Yemen': 'Asia',
    'Zambia': 'Africa', 'Zimbabwe': 'Africa'
}
df['continent'] = df['countrynew'].map(continent_mapping)

# Load ground truth
try:
    gt_df = pd.read_csv("data_final.csv")
    
    # Merge all available features
    feature_cols = ['countrynew', 'mean_age', 'mean_edu', 'mean_religion', 
                   'gdp_capita_2021', 'top1pct_income', 'top1pct_wealth', 
                   'hdi_2021', 'mean_temp_2010_2019', 'mean_own_willingness', 
                   'mean_other_willingness']
    
    # Only keep columns that exist
    available_cols = [col for col in feature_cols if col in gt_df.columns]
    gt_df = gt_df[available_cols]
    
    df = df.merge(gt_df, on='countrynew', how='left')
    
    # Create target variable
    df['ground_truth_pi'] = df['mean_other_willingness'] * 100
    
    print(f"✓ Loaded data: {len(df)} rows")
    print(f"✓ Countries: {df['countrynew'].nunique()}")
    print(f"✓ Continents: {sorted(df['continent'].dropna().unique())}")
    
except Exception as e:
    print(f"\n⚠️  Could not load ground truth: {e}")
    print("   Exiting...")
    exit()

# ================================================================
# 2. Prepare Features for OLS and Lasso
# ================================================================

print("\n" + "="*80)
print("2. PREPARING FEATURES")
print("="*80)

# Get unique country-level data (one row per country)
country_df = df.groupby('countrynew').first().reset_index()

# Define traditional features (excluding own_willingness to avoid data leakage)
traditional_features = ['mean_age', 'mean_edu', 'mean_religion', 
                       'gdp_capita_2021', 'top1pct_income', 'top1pct_wealth', 
                       'hdi_2021', 'mean_temp_2010_2019']

# Check which features are available
available_features = [f for f in traditional_features if f in country_df.columns 
                     and country_df[f].notna().sum() > 0]

print(f"\nTraditional features for OLS/Lasso: {len(available_features)}")
for f in available_features:
    n_missing = country_df[f].isna().sum()
    print(f"   - {f}: {len(country_df) - n_missing}/{len(country_df)} available")

# LLM predictions (we'll use Stage 8 - most information)
stage8_df = df[df['stage'] == 8].copy()
stage8_country = stage8_df.groupby('countrynew').first().reset_index()

llm_features = ['pred_gpt', 'pred_claude', 'pred_gemini', 'pred_llama', 'pred_ensemble']

print(f"\nLLM predictions (Stage 8): {len(llm_features)}")
for f in llm_features:
    if f in stage8_country.columns:
        n_available = stage8_country[f].notna().sum()
        print(f"   - {f}: {n_available}/{len(stage8_country)} available")

# Merge everything
analysis_df = country_df[['countrynew', 'continent', 'ground_truth_pi'] + available_features].copy()
analysis_df = analysis_df.merge(
    stage8_country[['countrynew'] + llm_features], 
    on='countrynew', 
    how='left'
)

# Remove rows with missing target
analysis_df = analysis_df[analysis_df['ground_truth_pi'].notna()].copy()

# Remove rows with missing features (for fair comparison)
analysis_df = analysis_df.dropna(subset=available_features + llm_features)

print(f"\n✓ Final dataset: {len(analysis_df)} countries with complete data")
print(f"   Target: ground_truth_pi (mean belief about others)")
print(f"   Features: {len(available_features)} traditional + {len(llm_features)} LLM predictions")

# ================================================================
# 3. Stratified Train-Test Splits (10 iterations)
# ================================================================

print("\n" + "="*80)
print("3. REPEATED TRAIN-TEST SPLITS")
print("="*80)

n_iterations = 10
test_size = 0.2
random_seeds = list(range(42, 42 + n_iterations))

print(f"\nRunning {n_iterations} iterations with different random seeds")
print(f"   Train size: {int((1-test_size)*100)}%")
print(f"   Test size: {int(test_size*100)}%")
print(f"   Stratification: By continent")

# Storage for results
all_results = []

# ================================================================
# 4. Run Iterations
# ================================================================

print("\n" + "="*80)
print("4. RUNNING MODELS")
print("="*80)

for iteration, seed in enumerate(random_seeds, 1):
    print(f"\n{'='*60}")
    print(f"Iteration {iteration}/{n_iterations} (seed={seed})")
    print('='*60)
    
    # Stratified split by continent
    train_df, test_df = train_test_split(
        analysis_df, 
        test_size=test_size, 
        random_state=seed,
        stratify=analysis_df['continent']
    )
    
    print(f"   Train: {len(train_df)} countries")
    print(f"   Test: {len(test_df)} countries")
    
    # Check continent distribution
    print("\n   Continent distribution in test set:")
    for continent in sorted(test_df['continent'].unique()):
        n = (test_df['continent'] == continent).sum()
        print(f"      {continent}: {n} countries")
    
    # Prepare data
    X_train_traditional = train_df[available_features].values
    X_test_traditional = test_df[available_features].values
    y_train = train_df['ground_truth_pi'].values
    y_test = test_df['ground_truth_pi'].values
    
    # Standardize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_traditional)
    X_test_scaled = scaler.transform(X_test_traditional)
    
    # ----------------------------------------------------------------
    # Model 1: OLS Regression
    # ----------------------------------------------------------------
    
    print("\n   [1] OLS Regression...")
    ols_model = LinearRegression()
    ols_model.fit(X_train_scaled, y_train)
    
    ols_train_pred = ols_model.predict(X_train_scaled)
    ols_test_pred = ols_model.predict(X_test_scaled)
    
    ols_train_mae = mean_absolute_error(y_train, ols_train_pred)
    ols_test_mae = mean_absolute_error(y_test, ols_test_pred)
    ols_train_rmse = np.sqrt(mean_squared_error(y_train, ols_train_pred))
    ols_test_rmse = np.sqrt(mean_squared_error(y_test, ols_test_pred))
    ols_train_r2 = r2_score(y_train, ols_train_pred)
    ols_test_r2 = r2_score(y_test, ols_test_pred)
    
    print(f"      Train MAE: {ols_train_mae:.2f}pp, RMSE: {ols_train_rmse:.2f}pp, R²: {ols_train_r2:.3f}")
    print(f"      Test MAE: {ols_test_mae:.2f}pp, RMSE: {ols_test_rmse:.2f}pp, R²: {ols_test_r2:.3f}")
    
    # ----------------------------------------------------------------
    # Model 2: Lasso Regression
    # ----------------------------------------------------------------
    
    print("\n   [2] Lasso Regression (with CV for alpha)...")
    lasso_cv = LassoCV(cv=5, random_state=seed, max_iter=10000)
    lasso_cv.fit(X_train_scaled, y_train)
    
    print(f"      Optimal alpha: {lasso_cv.alpha_:.4f}")
    
    lasso_train_pred = lasso_cv.predict(X_train_scaled)
    lasso_test_pred = lasso_cv.predict(X_test_scaled)
    
    lasso_train_mae = mean_absolute_error(y_train, lasso_train_pred)
    lasso_test_mae = mean_absolute_error(y_test, lasso_test_pred)
    lasso_train_rmse = np.sqrt(mean_squared_error(y_train, lasso_train_pred))
    lasso_test_rmse = np.sqrt(mean_squared_error(y_test, lasso_test_pred))
    lasso_train_r2 = r2_score(y_train, lasso_train_pred)
    lasso_test_r2 = r2_score(y_test, lasso_test_pred)
    
    print(f"      Train MAE: {lasso_train_mae:.2f}pp, RMSE: {lasso_train_rmse:.2f}pp, R²: {lasso_train_r2:.3f}")
    print(f"      Test MAE: {lasso_test_mae:.2f}pp, RMSE: {lasso_test_rmse:.2f}pp, R²: {lasso_test_r2:.3f}")
    
    # ----------------------------------------------------------------
    # Model 3-7: LLM Predictions (no training needed)
    # ----------------------------------------------------------------
    
    print("\n   [3-7] LLM Predictions...")
    
    llm_results = {}
    
    for llm_col in llm_features:
        llm_name = llm_col.replace('pred_', '').upper()
        
        # LLMs already made predictions - just evaluate
        llm_train_pred = train_df[llm_col].values
        llm_test_pred = test_df[llm_col].values
        
        llm_train_mae = mean_absolute_error(y_train, llm_train_pred)
        llm_test_mae = mean_absolute_error(y_test, llm_test_pred)
        llm_train_rmse = np.sqrt(mean_squared_error(y_train, llm_train_pred))
        llm_test_rmse = np.sqrt(mean_squared_error(y_test, llm_test_pred))
        llm_train_r2 = r2_score(y_train, llm_train_pred)
        llm_test_r2 = r2_score(y_test, llm_test_pred)
        
        print(f"      {llm_name}: Test MAE = {llm_test_mae:.2f}pp, RMSE = {llm_test_rmse:.2f}pp, R² = {llm_test_r2:.3f}")
        
        llm_results[llm_name] = {
            'train_mae': llm_train_mae,
            'test_mae': llm_test_mae,
            'train_rmse': llm_train_rmse,
            'test_rmse': llm_test_rmse,
            'train_r2': llm_train_r2,
            'test_r2': llm_test_r2,
            'train_pred': llm_train_pred,
            'test_pred': llm_test_pred
        }
    
    # ----------------------------------------------------------------
    # Store results for this iteration
    # ----------------------------------------------------------------
    
    iteration_results = {
        'iteration': iteration,
        'seed': seed,
        'n_train': len(train_df),
        'n_test': len(test_df),
        
        # OLS
        'ols_train_mae': ols_train_mae,
        'ols_test_mae': ols_test_mae,
        'ols_train_rmse': ols_train_rmse,
        'ols_test_rmse': ols_test_rmse,
        'ols_train_r2': ols_train_r2,
        'ols_test_r2': ols_test_r2,
        
        # Lasso
        'lasso_train_mae': lasso_train_mae,
        'lasso_test_mae': lasso_test_mae,
        'lasso_train_rmse': lasso_train_rmse,
        'lasso_test_rmse': lasso_test_rmse,
        'lasso_train_r2': lasso_train_r2,
        'lasso_test_r2': lasso_test_r2,
        'lasso_alpha': lasso_cv.alpha_,
    }
    
    # Add LLM results
    for llm_name, metrics in llm_results.items():
        iteration_results[f'{llm_name.lower()}_train_mae'] = metrics['train_mae']
        iteration_results[f'{llm_name.lower()}_test_mae'] = metrics['test_mae']
        iteration_results[f'{llm_name.lower()}_train_rmse'] = metrics['train_rmse']
        iteration_results[f'{llm_name.lower()}_test_rmse'] = metrics['test_rmse']
        iteration_results[f'{llm_name.lower()}_train_r2'] = metrics['train_r2']
        iteration_results[f'{llm_name.lower()}_test_r2'] = metrics['test_r2']
    
    all_results.append(iteration_results)

# ================================================================
# 5. Aggregate Results
# ================================================================

print("\n" + "="*80)
print("5. AGGREGATE RESULTS")
print("="*80)

results_df = pd.DataFrame(all_results)

# Calculate means and SDs across iterations
models = ['ols', 'lasso', 'gpt', 'claude', 'gemini', 'llama', 'ensemble']

summary_stats = []

for model in models:
    train_mae_col = f'{model}_train_mae'
    test_mae_col = f'{model}_test_mae'
    train_rmse_col = f'{model}_train_rmse'
    test_rmse_col = f'{model}_test_rmse'
    train_r2_col = f'{model}_train_r2'
    test_r2_col = f'{model}_test_r2'
    
    summary_stats.append({
        'Model': model.upper(),
        'Train MAE (Mean)': results_df[train_mae_col].mean(),
        'Train MAE (SD)': results_df[train_mae_col].std(),
        'Test MAE (Mean)': results_df[test_mae_col].mean(),
        'Test MAE (SD)': results_df[test_mae_col].std(),
        'Train RMSE (Mean)': results_df[train_rmse_col].mean(),
        'Train RMSE (SD)': results_df[train_rmse_col].std(),
        'Test RMSE (Mean)': results_df[test_rmse_col].mean(),
        'Test RMSE (SD)': results_df[test_rmse_col].std(),
        'Train R² (Mean)': results_df[train_r2_col].mean(),
        'Train R² (SD)': results_df[train_r2_col].std(),
        'Test R² (Mean)': results_df[test_r2_col].mean(),
        'Test R² (SD)': results_df[test_r2_col].std(),
    })

summary_df = pd.DataFrame(summary_stats)
summary_df = summary_df.sort_values('Test MAE (Mean)')

print("\n" + "="*60)
print("SUMMARY: Mean ± SD across 10 iterations")
print("="*60)
print("\nTest Set Performance (Primary Metric):")
print(summary_df[['Model', 'Test MAE (Mean)', 'Test MAE (SD)', 'Test RMSE (Mean)', 'Test RMSE (SD)', 'Test R² (Mean)', 'Test R² (SD)']].to_string(index=False))

print("\n\nTrain Set Performance (For comparison):")
print(summary_df[['Model', 'Train MAE (Mean)', 'Train MAE (SD)', 'Train RMSE (Mean)', 'Train RMSE (SD)', 'Train R² (Mean)', 'Train R² (SD)']].to_string(index=False))

# ================================================================
# 6. Statistical Comparison
# ================================================================

print("\n" + "="*80)
print("6. STATISTICAL COMPARISONS")
print("="*80)

# Pairwise t-tests comparing test MAE
print("\nPairwise comparisons (Test MAE):")
print("(Paired t-test for difference in means across 10 iterations)\n")

comparisons = [
    ('ENSEMBLE', 'OLS'),
    ('ENSEMBLE', 'LASSO'),
    ('LLAMA', 'OLS'),
    ('LLAMA', 'LASSO'),
    ('CLAUDE', 'OLS'),
    ('CLAUDE', 'LASSO'),
]

for model1, model2 in comparisons:
    col1 = f'{model1.lower()}_test_mae'
    col2 = f'{model2.lower()}_test_mae'
    
    t_stat, p_value = ttest_rel(results_df[col1], results_df[col2])
    
    mean_diff = results_df[col1].mean() - results_df[col2].mean()
    
    sig = ""
    if p_value < 0.001:
        sig = "***"
    elif p_value < 0.01:
        sig = "**"
    elif p_value < 0.05:
        sig = "*"
    
    print(f"   {model1} vs {model2}:")
    print(f"      Mean difference: {mean_diff:+.2f}pp")
    print(f"      t({len(results_df)-1}) = {t_stat:.3f}, p = {p_value:.4f} {sig}")

print("\n   * p < .05, ** p < .01, *** p < .001")

# ================================================================
# 7. Visualizations
# ================================================================

print("\n" + "="*80)
print("7. CREATING VISUALIZATIONS")
print("="*80)

# Figure 1: Main comparison with MAE and RMSE
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Predicting Pluralistic Ignorance: LLMs vs Traditional ML (Actual Data)', 
             fontsize=16, fontweight='bold')

# Panel A: Test MAE boxplot
ax = axes[0, 0]
test_mae_data = [results_df[f'{m}_test_mae'].values for m in models]

# Create better labels that indicate data source
label_mapping = {
    'ols': 'OLS\n(actual)',
    'lasso': 'LASSO\n(actual)',
    'llama': 'LLAMA',
    'gpt': 'GPT',
    'claude': 'CLAUDE',
    'gemini': 'GEMINI',
    'ensemble': 'ENSEMBLE'
}
plot_labels = [label_mapping[m] for m in models]

bp = ax.boxplot(test_mae_data, labels=plot_labels, patch_artist=True)

# Color code: OLS/Lasso vs LLMs
colors = ['#ff9999', '#ff9999', '#9999ff', '#9999ff', '#9999ff', '#9999ff', '#9999ff']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Test MAE (pp)', fontweight='bold')
ax.set_title('(a) Test Set Mean Absolute Error', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#ff9999', alpha=0.7, label='Traditional ML (actual data)'),
    Patch(facecolor='#9999ff', alpha=0.7, label='LLMs (Stage 8 predictions)')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)

# Panel B: Test RMSE boxplot
ax = axes[0, 1]
test_rmse_data = [results_df[f'{m}_test_rmse'].values for m in models]
bp = ax.boxplot(test_rmse_data, labels=plot_labels, patch_artist=True)

colors = ['#ff9999', '#ff9999', '#9999ff', '#9999ff', '#9999ff', '#9999ff', '#9999ff']
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel('Test RMSE (pp)', fontweight='bold')
ax.set_title('(b) Test Set Root Mean Squared Error', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Panel C: Test R² boxplot
ax = axes[0, 2]
test_r2_data = [results_df[f'{m}_test_r2'].values for m in models]
bp = ax.boxplot(test_r2_data, labels=plot_labels, patch_artist=True)

for patch in bp['boxes']:
    patch.set_facecolor('lightgreen')
    patch.set_alpha(0.7)

ax.set_ylabel('Test R²', fontweight='bold')
ax.set_title('(c) Test Set R² (Variance Explained)', fontweight='bold')
ax.axhline(y=0, color='red', linestyle='--', linewidth=1, alpha=0.5)
ax.grid(True, alpha=0.3, axis='y')
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Panel D: Mean Test MAE with error bars
ax = axes[1, 0]
means = [summary_df[summary_df['Model'] == m.upper()]['Test MAE (Mean)'].values[0] for m in models]
stds = [summary_df[summary_df['Model'] == m.upper()]['Test MAE (SD)'].values[0] for m in models]

colors = ['#d62728', '#d62728', '#1f77b4', '#1f77b4', '#1f77b4', '#1f77b4', '#1f77b4']
bars = ax.bar(range(len(models)), means, yerr=stds, capsize=5, alpha=0.7, color=colors)

ax.set_xticks(range(len(models)))
ax.set_xticklabels(plot_labels, rotation=45, ha='right')
ax.set_ylabel('Mean Test MAE (pp) ± SD', fontweight='bold')
ax.set_title('(d) Average MAE Across 10 Iterations', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add legend
legend_elements = [
    Patch(facecolor='#d62728', alpha=0.7, label='Traditional ML (actual data)'),
    Patch(facecolor='#1f77b4', alpha=0.7, label='LLMs (Stage 8 predictions)')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)

# Panel E: Mean Test RMSE with error bars
ax = axes[1, 1]
means_rmse = [summary_df[summary_df['Model'] == m.upper()]['Test RMSE (Mean)'].values[0] for m in models]
stds_rmse = [summary_df[summary_df['Model'] == m.upper()]['Test RMSE (SD)'].values[0] for m in models]

colors = ['#d62728', '#d62728', '#1f77b4', '#1f77b4', '#1f77b4', '#1f77b4', '#1f77b4']
bars = ax.bar(range(len(models)), means_rmse, yerr=stds_rmse, capsize=5, alpha=0.7, color=colors)

ax.set_xticks(range(len(models)))
ax.set_xticklabels(plot_labels, rotation=45, ha='right')
ax.set_ylabel('Mean Test RMSE (pp) ± SD', fontweight='bold')
ax.set_title('(e) Average RMSE Across 10 Iterations', fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Panel F: Train vs Test (overfitting check) - using MAE
ax = axes[1, 2]
for i, model in enumerate(models):
    train_mae = summary_df[summary_df['Model'] == model.upper()]['Train MAE (Mean)'].values[0]
    test_mae = summary_df[summary_df['Model'] == model.upper()]['Test MAE (Mean)'].values[0]
    
    color = '#d62728' if model in ['ols', 'lasso'] else '#1f77b4'
    ax.scatter(train_mae, test_mae, s=100, alpha=0.7, color=color)
    
    # Use the annotated labels
    label_text = plot_labels[i].replace('\n', ' ')
    ax.text(train_mae, test_mae, label_text, fontsize=8, ha='right', va='bottom')

# Perfect generalization line
max_val = max(ax.get_xlim()[1], ax.get_ylim()[1])
ax.plot([0, max_val], [0, max_val], 'k--', linewidth=1, alpha=0.5, label='Perfect generalization')

ax.set_xlabel('Train MAE (pp)', fontweight='bold')
ax.set_ylabel('Test MAE (pp)', fontweight='bold')
ax.set_title('(f) Generalization (Train vs Test Error)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ml_comparison_llm_vs_ols_vs_lasso.png', dpi=300, bbox_inches='tight')
plt.savefig('ml_comparison_llm_vs_ols_vs_lasso.pdf', dpi=300, bbox_inches='tight')
print("\n✓ Saved ml_comparison_llm_vs_ols_vs_lasso.png (2×3 grid with MAE, RMSE, R²)")
print("✓ Saved ml_comparison_llm_vs_ols_vs_lasso.pdf (2×3 grid with MAE, RMSE, R²)")
plt.close()

# ================================================================
# 8. Export Results
# ================================================================

print("\n" + "="*80)
print("8. EXPORTING RESULTS")
print("="*80)

# Save detailed results
results_df.to_csv('ml_detailed_results_10iterations.csv', index=False)
print("✓ Saved ml_detailed_results_10iterations.csv")

# Save summary
summary_df.to_csv('ml_summary_comparison.csv', index=False)
print("✓ Saved ml_summary_comparison.csv")

# ================================================================
# 9. Final Summary
# ================================================================

print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)

best_model = summary_df.iloc[0]['Model']
best_mae = summary_df.iloc[0]['Test MAE (Mean)']
best_r2 = summary_df.iloc[0]['Test R² (Mean)']

print(f"\nBest performing model: {best_model}")
print(f"   Test MAE: {best_mae:.2f} ± {summary_df.iloc[0]['Test MAE (SD)']:.2f}pp")
print(f"   Test R²: {best_r2:.3f} ± {summary_df.iloc[0]['Test R² (SD)']:.3f}")

# Comparison summary
ols_mae = summary_df[summary_df['Model'] == 'OLS']['Test MAE (Mean)'].values[0]
lasso_mae = summary_df[summary_df['Model'] == 'LASSO']['Test MAE (Mean)'].values[0]
ensemble_mae = summary_df[summary_df['Model'] == 'ENSEMBLE']['Test MAE (Mean)'].values[0]

print(f"\nKey Comparisons:")
print(f"   LLM Ensemble vs OLS: {ensemble_mae - ols_mae:+.2f}pp difference")
print(f"   LLM Ensemble vs Lasso: {ensemble_mae - lasso_mae:+.2f}pp difference")

if ensemble_mae < ols_mae:
    print(f"\n✅ LLMs outperform traditional regression by {ols_mae - ensemble_mae:.2f}pp!")
elif ensemble_mae < lasso_mae:
    print(f"\n✅ LLMs outperform Lasso regression by {lasso_mae - ensemble_mae:.2f}pp!")
else:
    print(f"\n⚠️  Traditional ML performs better, but LLMs still competitive")

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)
print("\nFiles created:")
print("  - ml_comparison_llm_vs_ols_vs_lasso.png")
print("  - ml_comparison_llm_vs_ols_vs_lasso.pdf")
print("  - ml_detailed_results_10iterations.csv")
print("  - ml_summary_comparison.csv")

ML COMPARISON: LLMs (Stage 8) vs OLS vs LASSO (Actual Data)

Comparing three approaches to predict pluralistic ignorance:
  1. LLM predictions from Stage 8 (GPT, Claude, Gemini, Llama, Ensemble)
  2. OLS regression trained on actual country features & outcomes
  3. Lasso regression trained on actual country features & outcomes

Evaluation: 10 repeated 80:20 train-test splits (stratified by continent)

1. LOADING DATA
✓ Loaded data: 1000 rows
✓ Countries: 125
✓ Continents: ['Africa', 'Asia', 'Europe', 'North America', 'Oceania', 'South America']

2. PREPARING FEATURES

Traditional features for OLS/Lasso: 7
   - mean_age: 122/125 available
   - mean_edu: 122/125 available
   - mean_religion: 116/125 available
   - gdp_capita_2021: 125/125 available
   - top1pct_income: 125/125 available
   - top1pct_wealth: 125/125 available
   - hdi_2021: 123/125 available

LLM predictions (Stage 8): 5
   - pred_gpt: 125/125 available
   - pred_claude: 125/125 available
   - pred_gemini: 125/125 availab

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>